# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:
!pip install -q faiss-cpu google-genai

import pandas as pd
import numpy as np
import faiss
from google import genai
from google.genai import types
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

df = pd.read_csv('ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [2]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [4]:
def get_embedding(text):
  result = client.models.embed_content(
      model='gemini-embedding-001',  # Ganti dari text-embedding-004
      contents=text,
  )
  return result.embeddings[0].values


embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [5]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

Jumlah vektor tersimpan di index: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [6]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.33353508)}
{'rank': 2, 'question': 'Kenapa status pengiriman saya tidak berubah selama beberapa hari?', 'answer': 'Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.', 'category': 'Pengiriman', 'distance': np.float32(0.4088098)}
{'rank': 3, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.4320181)}


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [7]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
{context}

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

## 7. Mengintegrasikan LLM

In [8]:
def generate_answer(query, top_k=3):
  search_results = semantic_search(query, top_k=top_k)
  prompt = build_prompt(query, search_results)

  response = client.models.generate_content(
      model='gemini-3.5-flash-lite',  # Ganti dari gemini-2.0-flash
      contents=prompt,
      config=types.GenerateContentConfig(temperature=0.2),
  )
  return response.text, search_results


answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
  print('-', s['question'])

Jawaban LLM:
Halo! Tentu, fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank. Ada hal lain yang bisa saya bantu?

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [9]:
test_queries = [
    # 1. Direct Match (Jelas ada di FAQ - Topik Pemesanan/Pembatalan)
    'Bagaimana cara membatalkan pesanan yang sudah dibuat?',
    # 2. Direct Match (Jelas ada di FAQ - Topik Akun)
    'Bagaimana cara menghapus akun secara permanen?',
    # 3. Semantic Paraphrase / Sinonim (Tidak pakai kata 'estimasi' atau 'resi', tapi makna sama)
    'Barang belanjaan saya kok belum nyampe-nyampe ya padahal sudah lewat jadwalnya?',
    # 4. Semantic Paraphrase / Bahasa Sehari-hari (Menanyakan ganti alamat setelah checkout)
    'Salah masukin alamat rumah nih, masih bisa diubah nggak kalau sudah terlanjur bayar?',
    # 5. Out-of-Domain / Tidak ada di FAQ (Menguji Grounding & Hallucination Control)
    'Apakah platform ini menyediakan layanan pinjaman dana tunai atau paylater untuk beli motor?',
    # 6. Out-of-Domain / Di luar konteks e-commerce
    'Siapa pemenang Piala Dunia tahun 2022?',
]

for i, q in enumerate(test_queries, 1):
  print(f'=== TEST CASE #{i} ===')
  print('Query:', q)
  answer, sources = generate_answer(q, top_k=3)
  print('Jawaban LLM:\n', answer.strip())
  print('\nTop-3 Retrieved Contexts (L2 Distance):')
  for s in sources:
    print(
        f"  [{s['rank']}] (Dist: {s['distance']:.4f}) [{s['category']}]"
        f" {s['question']}"
    )
  print('=' * 50 + '\n')

=== TEST CASE #1 ===
Query: Bagaimana cara membatalkan pesanan yang sudah dibuat?
Jawaban LLM:
 Halo! Pesanan dapat dibatalkan melalui halaman 'Pesanan Saya' selama status masih 'Diproses'. Namun, setelah status berubah menjadi 'Dikirim', pembatalan tidak dapat dilakukan. 

Ada hal lain yang bisa saya bantu?

Top-3 Retrieved Contexts (L2 Distance):
  [1] (Dist: 0.2392) [Pemesanan] Bagaimana cara membatalkan pesanan yang sudah dibuat?
  [2] (Dist: 0.5069) [Pengembalian] Bagaimana prosedur pengembalian barang yang tidak sesuai?
  [3] (Dist: 0.5349) [Produk] Bagaimana jika produk yang saya pesan ternyata kehabisan stok?

=== TEST CASE #2 ===
Query: Bagaimana cara menghapus akun secara permanen?
Jawaban LLM:
 Halo! Tentu, saya akan bantu. 

Pengajuan penghapusan akun secara permanen dapat dilakukan melalui menu 'Pengaturan Akun > Hapus Akun' atau dengan menghubungi customer service kami. 

Ada hal lain yang bisa saya bantu?

Top-3 Retrieved Contexts (L2 Distance):
  [1] (Dist: 0.1889) [Aku

In [10]:
%%writefile retrieval_test_results.md
from google.colab import files
files.download('retrieval_test_results.md')

=== TEST CASE #1 ===
Query: Bagaimana cara membatalkan pesanan yang sudah dibuat?
Jawaban LLM:
 Halo! Pesanan dapat dibatalkan melalui halaman 'Pesanan Saya' selama status masih 'Diproses'. Namun, setelah status berubah menjadi 'Dikirim', pembatalan sudah tidak dapat dilakukan.

Ada lagi yang bisa saya bantu?

Top-3 Retrieved Contexts (L2 Distance):
  [1] (Dist: 0.2392) [Pemesanan] Bagaimana cara membatalkan pesanan yang sudah dibuat?
  [2] (Dist: 0.5069) [Pengembalian] Bagaimana prosedur pengembalian barang yang tidak sesuai?
  [3] (Dist: 0.5349) [Produk] Bagaimana jika produk yang saya pesan ternyata kehabisan stok?
==================================================

=== TEST CASE #2 ===
Query: Bagaimana cara menghapus akun secara permanen?
Jawaban LLM:
 Halo! Untuk menghapus akun secara permanen, pengajuan dapat dilakukan melalui menu 'Pengaturan Akun > Hapus Akun' atau dengan menghubungi customer service kami. Ada hal lain yang bisa saya bantu?

Top-3 Retrieved Contexts (L2 Distance):
  [1] (Dist: 0.1889) [Akun] Bagaimana cara menghapus akun secara permanen?
  [2] (Dist: 0.6851) [Akun] Apakah bisa memiliki lebih dari satu akun dengan email yang sama?
  [3] (Dist: 0.6976) [Akun] Saya lupa password akun, bagaimana cara mengaturnya ulang?
==================================================

=== TEST CASE #3 ===
Query: Barang belanjaan saya kok belum nyampe-nyampe ya padahal sudah lewat jadwalnya?
Jawaban LLM:
 Halo! Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.

Top-3 Retrieved Contexts (L2 Distance):
  [1] (Dist: 0.3777) [Pengiriman] Bagaimana jika paket saya belum sampai melebihi estimasi?
  [2] (Dist: 0.4145) [Pengiriman] Kenapa status pengiriman saya tidak berubah selama beberapa hari?
  [3] (Dist: 0.4830) [Pengiriman] Berapa lama estimasi waktu pengiriman barang?
==================================================

=== TEST CASE #4 ===
Query: Salah masukin alamat rumah nih, masih bisa diubah nggak kalau sudah terlanjur bayar?
Jawaban LLM:
 Halo! Perubahan alamat hanya bisa dilakukan sebelum pesanan berstatus 'Dikirim'. Silakan hubungi customer service secepatnya jika ingin mengubah alamat.

Top-3 Retrieved Contexts (L2 Distance):
  [1] (Dist: 0.2304) [Pemesanan] Apakah saya bisa mengubah alamat pengiriman setelah pesanan dibuat?
  [2] (Dist: 0.5809) [Pengembalian] Saya menerima barang yang salah, apa yang harus saya lakukan?
  [3] (Dist: 0.5873) [Pemesanan] Bagaimana cara membatalkan pesanan yang sudah dibuat?
==================================================

=== TEST CASE #5 ===
Query: Apakah platform ini menyediakan layanan pinjaman dana tunai atau paylater untuk beli motor?
Jawaban LLM:
 Halo! Mohon maaf, saya belum memiliki informasi tersebut. Untuk informasi lebih lanjut, silakan menghubungi customer service kami.

Top-3 Retrieved Contexts (L2 Distance):
  [1] (Dist: 0.6429) [Pembayaran] Apakah saya bisa membayar dengan cicilan?
  [2] (Dist: 0.7287) [Pembayaran] Metode pembayaran apa saja yang tersedia?
  [3] (Dist: 0.7339) [Pembayaran] Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
==================================================

=== TEST CASE #6 ===
Query: Siapa pemenang Piala Dunia tahun 2022?
Jawaban LLM:
 Halo! Maaf sekali, saya belum memiliki informasi tersebut. Silakan hubungi customer service kami untuk bantuan lebih lanjut ya. Ada hal lain seputar pesanan atau e-commerce kami yang bisa saya bantu?

Top-3 Retrieved Contexts (L2 Distance):
  [1] (Dist: 0.8974) [Pembayaran] Metode pembayaran apa saja yang tersedia?
  [2] (Dist: 0.9164) [Pembayaran] Kenapa pembayaran saya berhasil dipotong tapi status pesanan masih belum terbayar?
  [3] (Dist: 0.9258) [Pembayaran] Berapa lama batas waktu pembayaran sebelum pesanan dibatalkan?
==================================================

---

# Catatan Analisis Relevansi Sistem RAG (Semantic Search + LLM)



1. **Evaluasi Retrieval Quality pada Direct Query & Semantic Paraphrase (Sinonim)**:
	- Hasil: Sistem berhasil melakukan dense retrieval dengan akurasi sangat tinggi baik pada query eksplisit maupun query parafrase/bahasa sehari-hari (misalnya frasa "barang belum nyampe lewat jadwal" tetap berhasil menarik FAQ "Bagaimana jika paket saya belum sampai melebihi estimasi?" di peringkat #1 dengan skor L2 Distance terkecil/paling dekat).
	- Penjelasan : Model gemini-embedding-001 memetakan teks ke dalam high-dimensional vector space (3072 dimensi) berdasarkan makna semantik (semantic intent), bukan pencocokan leksikal (exact keyword matching seperti TF-IDF atau SQL LIKE). Karena unit dokumen menggabungkan format Q: ... A: ..., vektor dokumen menjadi lebih kaya konteks (context-rich representation).
	-  Analogi Sederhana: Bayangkan seorang pustakawan cerdas. Pustakawan biasa (keyword search) hanya mencari buku berdasarkan judul yang persis sama dengan kata yang Anda ucapkan. Sebaliknya, pustakawan cerdas (semantic search) memahami maksud di balik keluhan user—meski user memakai bahasa santai, ia langsung tahu rak mana yang menyimpan solusinya.

---

2. **Evaluasi Grounding & Hallucination Control pada Out-of-Domain Query**:
	- Hasil: Ketika diuji dengan pertanyaan di luar knowledge base (seperti pertanyaan soal paylater motor atau Piala Dunia), LLM (gemini-3.5-flash-lite) mematuhi system instruction dengan menolak menjawab secara mengarang dan menyarankan pengguna menghubungi customer service.
	- Penjelasan: Ini membuktikan efektivitas prompt grounding dan pengaturan temperature=0.2 (rendah/deterministik). Meskipun FAISS tetap mengembalikan Top-3 dokumen terdekat secara matematis (forced retrieval), LLM bertindak sebagai reasoning filter yang mengevaluasi bahwa konteks yang diberikan tidak relevan dengan query pengguna, sehingga mencegah halusinasi (hallucination).
	- Analogi Sederhana: Bayangkan seorang staf CS yang dibekali buku panduan SOP. Saat pelanggan bertanya hal di luar buku SOP, mesin pencari tetap menyodorkan 3 halaman buku yang paling menyerupai pertanyaan tersebut. Namun, karena staf CS dilatih disiplin (temperature=0.2 dan aturan grounding), ia membaca halaman itu, sadar bahwa jawabannya tidak ada di sana, lalu dengan jujur berkata "Maaf, informasi tersebut belum tersedia di panduan kami" alih-alih mengarang aturan sendiri.

---

3. **Temuan Potensial Noise pada Top-K Retrieval**:
	- Hasil: Karena fungsi semantic_search selalu mengambil top_k=3 tanpa batas ambang jarak (distance threshold), untuk pertanyaan yang topiknya sangat spesifik (misalnya penghapusan akun), peringkat #2 dan #3 terkadang berisi FAQ yang agak berjauhan topiknya namun masih satu kategori.
	- Analogi Sederhana: Seperti meminta asisten membawakan 3 berkas paling mirip di meja. Jika hanya ada 1 berkas yang benar-benar cocok, asisten tersebut tetap terpaksa mengambil 2 berkas tambahan di sebelahnya sekadar untuk memenuhi kuota 3 berkas.



Overwriting retrieval_test_results.md


# Catatan Analisis Relevansi Sistem RAG (Semantic Search + LLM)



1. **Evaluasi Retrieval Quality pada Direct Query & Semantic Paraphrase (Sinonim)**:
	- Hasil: Sistem berhasil melakukan dense retrieval dengan akurasi sangat tinggi baik pada query eksplisit maupun query parafrase/bahasa sehari-hari (misalnya frasa "barang belum nyampe lewat jadwal" tetap berhasil menarik FAQ "Bagaimana jika paket saya belum sampai melebihi estimasi?" di peringkat #1 dengan skor L2 Distance terkecil/paling dekat).
	- Penjelasan : Model gemini-embedding-001 memetakan teks ke dalam high-dimensional vector space (3072 dimensi) berdasarkan makna semantik (semantic intent), bukan pencocokan leksikal (exact keyword matching seperti TF-IDF atau SQL LIKE). Karena unit dokumen menggabungkan format Q: ... A: ..., vektor dokumen menjadi lebih kaya konteks (context-rich representation).
	-  Analogi Sederhana: Bayangkan seorang pustakawan cerdas. Pustakawan biasa (keyword search) hanya mencari buku berdasarkan judul yang persis sama dengan kata yang Anda ucapkan. Sebaliknya, pustakawan cerdas (semantic search) memahami maksud di balik keluhan user—meski user memakai bahasa santai, ia langsung tahu rak mana yang menyimpan solusinya.

---

2. **Evaluasi Grounding & Hallucination Control pada Out-of-Domain Query**:
	- Hasil: Ketika diuji dengan pertanyaan di luar knowledge base (seperti pertanyaan soal paylater motor atau Piala Dunia), LLM (gemini-3.5-flash-lite) mematuhi system instruction dengan menolak menjawab secara mengarang dan menyarankan pengguna menghubungi customer service.
	- Penjelasan: Ini membuktikan efektivitas prompt grounding dan pengaturan temperature=0.2 (rendah/deterministik). Meskipun FAISS tetap mengembalikan Top-3 dokumen terdekat secara matematis (forced retrieval), LLM bertindak sebagai reasoning filter yang mengevaluasi bahwa konteks yang diberikan tidak relevan dengan query pengguna, sehingga mencegah halusinasi (hallucination).
	- Analogi Sederhana: Bayangkan seorang staf CS yang dibekali buku panduan SOP. Saat pelanggan bertanya hal di luar buku SOP, mesin pencari tetap menyodorkan 3 halaman buku yang paling menyerupai pertanyaan tersebut. Namun, karena staf CS dilatih disiplin (temperature=0.2 dan aturan grounding), ia membaca halaman itu, sadar bahwa jawabannya tidak ada di sana, lalu dengan jujur berkata "Maaf, informasi tersebut belum tersedia di panduan kami" alih-alih mengarang aturan sendiri.

---

3. **Temuan Potensial Noise pada Top-K Retrieval**:
	- Hasil: Karena fungsi semantic_search selalu mengambil top_k=3 tanpa batas ambang jarak (distance threshold), untuk pertanyaan yang topiknya sangat spesifik (misalnya penghapusan akun), peringkat #2 dan #3 terkadang berisi FAQ yang agak berjauhan topiknya namun masih satu kategori.
	- Analogi Sederhana: Seperti meminta asisten membawakan 3 berkas paling mirip di meja. Jika hanya ada 1 berkas yang benar-benar cocok, asisten tersebut tetap terpaksa mengambil 2 berkas tambahan di sebelahnya sekadar untuk memenuhi kuota 3 berkas.


# README.md
## E-Commerce FAQ Semantic Search & RAG Assistant

- Prototype aplikasi tanya-jawab cerdas (*Question-Answering System*) berbasis arsitektur **Retrieval-Augmented Generation (RAG)** untuk layanan pelanggan *e-commerce*. Sistem ini menggabungkan pencarian berbasis makna semantik (**Gemini Embeddings + FAISS**) dengan generasi jawaban kontekstual (**Gemini LLM**).

---

## 1. Problem Statement & Knowledge Base

### Problem Statement
- Sistem pencarian FAQ konvensional yang mengandalkan pencocokan kata kunci (*keyword-based search*) sering gagal memahami pertanyaan pelanggan yang menggunakan sinonim, bahasa sehari-hari (*slang*), atau struktur kalimat yang berbeda dari dokumen resmi.
- Akibatnya, pelanggan kesulitan menemukan solusi mandiri dan beban tiket *customer service* meningkat. Proyek ini membangun sistem pencarian berbasis makna (*semantic search*) yang mampu memahami maksud (*intent*) pengguna dan menghasilkan jawaban natural yang akurat tanpa halusinasi.

### Knowledge Base
* **Sumber Data:** `ecommerce_faq_knowledge_base.csv`
* **Volume & Struktur Data:** Terdiri dari **59 entri FAQ** yang mencakup **8 kategori** utama layanan *e-commerce* (seperti *Pemesanan*, *Pengiriman*, *Pembayaran*, *Akun*, *Promo*, dll.).
* **Atribut Dataset:** `faq_id`, `category`, `question`, dan `answer`.

---

## 2. Arsitektur & Alur Sistem (RAG Pipeline)

Sistem bekerja melalui lima tahapan utama secara *end-to-end*:

```text
[CSV Knowledge Base] ──▶ (1. Ingest) ──▶ [Unit Retrievable (Q+A)]
                                                │
                                          (2. Embed) ──▶ Gemini Embedding API (3072-d)
                                                │
                                          (3. Store) ──▶ FAISS Vector Index (IndexFlatL2)
                                                │
[User Query] ──────────▶ (4. Retrieve) ─────────┴──▶ Top-K Relevant Contexts
                                                            │
                                                      (5. Generate) ──▶ Prompt + Gemini LLM ──▶ [Final Answer]
```

1. **Ingest:** Memuat data CSV menggunakan `pandas` dan menggabungkan kolom `question` serta `answer` menjadi satu unit teks utuh (`Q: {question} A: {answer}`) agar representasi maknanya kaya konteks.
2. **Embed:** Mengonversi setiap unit teks FAQ menjadi vektor numerik berdimensi **3072** menggunakan model **`gemini-embedding-001`** melalui Google GenAI SDK.
3. **Store:** Menyimpan seluruh vektor embedding ke dalam *in-memory vector database* **FAISS** menggunakan indeks **`IndexFlatL2`** (perhitungan jarak *Euclidean / L2 Distance*).
4. **Retrieve:** Ketika pengguna memasukkan *query*, sistem mengubah *query* menjadi vektor menggunakan model embedding yang sama, lalu mencari **Top-3 (`top_k=3`)** entri FAQ dengan jarak vektor terdekat.
5. **Generate:** Menyisipkan hasil *Top-3* FAQ ke dalam *structured prompt* (berisi *Persona*, *Task*, dan *Grounding Rules*), lalu mengirimkannya ke model **`gemini-3.5-flash-lite`** (`temperature=0.2`) untuk merangkum jawaban yang ramah dan faktual.

---

## 3. Hasil Pengujian Query & Analisis Relevansi

Pengujian dilakukan menggunakan beberapa kategori *query* untuk mengevaluasi ketahanan sistem:

| Kategori Uji | Contoh Query Pengguna | Hasil Retrieval (Top-1) | Evaluasi & Analisis Relevansi |
| :--- | :--- | :--- | :--- |
| **Direct Match** | *"Apakah saya bisa membayar dengan cicilan?"* | *Apakah saya bisa membayar dengan cicilan?* | **Sangat Relevan.** Sistem langsung menangkap entri yang tepat di peringkat #1 dan merangkum syarat kartu kredit dengan akurat. |
| **Semantic Paraphrase** | *"Kenapa paket saya belum sampai?"* / *"Barang belum nyampe lewat jadwal"* | *Bagaimana jika paket saya belum sampai melebihi estimasi?* (`Dist: 0.3335`) | **Sangat Relevan.** Meskipun pengguna tidak memakai kata *"estimasi"*, vektor embedding berhasil memetakan kesamaan makna keterlambatan pengiriman. |
| **Out-of-Domain (OOD)** | *"Apakah tersedia layanan pinjaman tunai / paylater motor?"* | *Metode pembayaran apa saja yang tersedia?* (Konteks terdekat yang dipaksakan) | **Grounding Berhasil.** LLM mendeteksi bahwa informasi tidak ada dalam konteks FAQ, menolak mengarang jawaban, dan mengarahkan pengguna ke CS. |

**Kesimpulan Analisis:** Kombinasi `gemini-embedding-001` dan `IndexFlatL2` sangat efektif menangkap variasi bahasa alami pengguna, sementara pengaturan `temperature=0.2` dan instruksi *strict context grounding* berhasil menekan risiko halusinasi LLM.

---

## 4. Keterbatasan Sistem (System Limitations)

1. **Ketiadaan *Distance Threshold* (Forced Top-K):** Fungsi `semantic_search` selalu mengembalikan 3 dokumen teratas tanpa filter batas jarak maksimum. Pada *query* di luar domain (*out-of-domain*), dokumen yang tidak relevan tetap ikut terkirim ke dalam *context window* LLM sehingga memboroskan token.
2. **Penyimpanan *In-Memory* (Non-Persisten):** Indeks FAISS disimpan di RAM runtime. Setiap kali sesi notebook di-restart, proses *embedding* ulang terhadap seluruh knowledge base harus dijalankan kembali.
3. **Pencarian Murni *Dense Vector* (Tanpa *Hybrid Search*):** Sistem belum menggabungkan pencarian leksikal (seperti BM25). Untuk istilah yang sangat spesifik seperti kode promo unik atau nomor seri tertentu, *pure dense search* terkadang kalah presisi dibanding pencarian kata kunci eksak.
4. **Stateless (Tanpa Memori Percakapan):** Sistem memproses setiap *query* secara independen (*single-turn*) dan belum menyimpan riwayat percakapan (*chat history*) untuk pertanyaan lanjutan (*follow-up questions*).

---

## 5. Cara Menjalankan Notebook

### Prasyarat
* Akun Google dengan akses ke **Google Colab**.
* **Gemini API Key** aktif yang didapatkan dari [Google AI Studio](https://aistudio.google.com/).
* File dataset `ecommerce_faq_knowledge_base.csv`.

### Langkah-Langkah Eksekusi
1. **Buka Notebook:** Unggah dan buka file `Copy of starter_notebook_semantic_search.ipynb` di Google Colab.
2. **Unggah Dataset:** Buka panel **Files** (ikon folder di bilah kiri Colab), lalu unggah file `ecommerce_faq_knowledge_base.csv` ke direktori utama (`/content/`).
3. **Konfigurasi API Key (Secrets):**
   * Klik ikon **Secrets (Kunci)** di bilah navigasi kiri Colab.
   * Tambahkan *secret* baru dengan nama `GEMINI_API_KEY` dan masukkan nilai API key Anda.
   * Aktifkan *toggle* **Notebook access**.
4. **Jalankan Sel Berurutan:**
   * Klik menu **Runtime > Run all** (atau jalankan setiap *code cell* secara berurutan dari Section 1 hingga Section 8 menggunakan `Shift + Enter`).
   * Sel pertama akan otomatis menginstal pustaka yang dibutuhkan (`faiss-cpu` dan `google-genai`) sebelum membangun pipeline pencarian.


In [11]:
%%writefile README.md
from google.colab import files
files.download('README.md')

Nama: SATRIO ADI WIBOWO
Absen: AI-70
Program: REWORK AI ENGINEERING BOOTCAMP

# E-Commerce FAQ Semantic Search & RAG Assistant

- Prototype aplikasi tanya-jawab cerdas (*Question-Answering System*) berbasis arsitektur **Retrieval-Augmented Generation (RAG)** untuk layanan pelanggan *e-commerce*. Sistem ini menggabungkan pencarian berbasis makna semantik (**Gemini Embeddings + FAISS**) dengan generasi jawaban kontekstual (**Gemini LLM**).

---

## 1. Problem Statement & Knowledge Base

### Problem Statement
- Sistem pencarian FAQ konvensional yang mengandalkan pencocokan kata kunci (*keyword-based search*) sering gagal memahami pertanyaan pelanggan yang menggunakan sinonim, bahasa sehari-hari (*slang*), atau struktur kalimat yang berbeda dari dokumen resmi.
- Akibatnya, pelanggan kesulitan menemukan solusi mandiri dan beban tiket *customer service* meningkat. Proyek ini membangun sistem pencarian berbasis makna (*semantic search*) yang mampu memahami maksud (*intent*) pengguna dan menghasilkan jawaban natural yang akurat tanpa halusinasi.

### Knowledge Base
* **Sumber Data:** `ecommerce_faq_knowledge_base.csv`
* **Volume & Struktur Data:** Terdiri dari **59 entri FAQ** yang mencakup **8 kategori** utama layanan *e-commerce* (seperti *Pemesanan*, *Pengiriman*, *Pembayaran*, *Akun*, *Promo*, dll.).
* **Atribut Dataset:** `faq_id`, `category`, `question`, dan `answer`.

---

## 2. Arsitektur & Alur Sistem (RAG Pipeline)

Sistem bekerja melalui lima tahapan utama secara *end-to-end*:

```text
[CSV Knowledge Base] ──▶ (1. Ingest) ──▶ [Unit Retrievable (Q+A)]
                                                │
                                          (2. Embed) ──▶ Gemini Embedding API (3072-d)
                                                │
                                          (3. Store) ──▶ FAISS Vector Index (IndexFlatL2)
                                                │
[User Query] ──────────▶ (4. Retrieve) ─────────┴──▶ Top-K Relevant Contexts
                                                            │
                                                      (5. Generate) ──▶ Prompt + Gemini LLM ──▶ [Final Answer]
```

1. **Ingest:** Memuat data CSV menggunakan `pandas` dan menggabungkan kolom `question` serta `answer` menjadi satu unit teks utuh (`Q: {question} A: {answer}`) agar representasi maknanya kaya konteks.
2. **Embed:** Mengonversi setiap unit teks FAQ menjadi vektor numerik berdimensi **3072** menggunakan model **`gemini-embedding-001`** melalui Google GenAI SDK.
3. **Store:** Menyimpan seluruh vektor embedding ke dalam *in-memory vector database* **FAISS** menggunakan indeks **`IndexFlatL2`** (perhitungan jarak *Euclidean / L2 Distance*).
4. **Retrieve:** Ketika pengguna memasukkan *query*, sistem mengubah *query* menjadi vektor menggunakan model embedding yang sama, lalu mencari **Top-3 (`top_k=3`)** entri FAQ dengan jarak vektor terdekat.
5. **Generate:** Menyisipkan hasil *Top-3* FAQ ke dalam *structured prompt* (berisi *Persona*, *Task*, dan *Grounding Rules*), lalu mengirimkannya ke model **`gemini-3.5-flash-lite`** (`temperature=0.2`) untuk merangkum jawaban yang ramah dan faktual.

---

## 3. Hasil Pengujian Query & Analisis Relevansi

Pengujian dilakukan menggunakan beberapa kategori *query* untuk mengevaluasi ketahanan sistem:

| Kategori Uji | Contoh Query Pengguna | Hasil Retrieval (Top-1) | Evaluasi & Analisis Relevansi |
| :--- | :--- | :--- | :--- |
| **Direct Match** | *"Apakah saya bisa membayar dengan cicilan?"* | *Apakah saya bisa membayar dengan cicilan?* | **Sangat Relevan.** Sistem langsung menangkap entri yang tepat di peringkat #1 dan merangkum syarat kartu kredit dengan akurat. |
| **Semantic Paraphrase** | *"Kenapa paket saya belum sampai?"* / *"Barang belum nyampe lewat jadwal"* | *Bagaimana jika paket saya belum sampai melebihi estimasi?* (`Dist: 0.3335`) | **Sangat Relevan.** Meskipun pengguna tidak memakai kata *"estimasi"*, vektor embedding berhasil memetakan kesamaan makna keterlambatan pengiriman. |
| **Out-of-Domain (OOD)** | *"Apakah tersedia layanan pinjaman tunai / paylater motor?"* | *Metode pembayaran apa saja yang tersedia?* (Konteks terdekat yang dipaksakan) | **Grounding Berhasil.** LLM mendeteksi bahwa informasi tidak ada dalam konteks FAQ, menolak mengarang jawaban, dan mengarahkan pengguna ke CS. |

**Kesimpulan Analisis:** Kombinasi `gemini-embedding-001` dan `IndexFlatL2` sangat efektif menangkap variasi bahasa alami pengguna, sementara pengaturan `temperature=0.2` dan instruksi *strict context grounding* berhasil menekan risiko halusinasi LLM.

---

## 4. Keterbatasan Sistem (System Limitations)

1. **Ketiadaan *Distance Threshold* (Forced Top-K):** Fungsi `semantic_search` selalu mengembalikan 3 dokumen teratas tanpa filter batas jarak maksimum. Pada *query* di luar domain (*out-of-domain*), dokumen yang tidak relevan tetap ikut terkirim ke dalam *context window* LLM sehingga memboroskan token.
2. **Penyimpanan *In-Memory* (Non-Persisten):** Indeks FAISS disimpan di RAM runtime. Setiap kali sesi notebook di-restart, proses *embedding* ulang terhadap seluruh knowledge base harus dijalankan kembali.
3. **Pencarian Murni *Dense Vector* (Tanpa *Hybrid Search*):** Sistem belum menggabungkan pencarian leksikal (seperti BM25). Untuk istilah yang sangat spesifik seperti kode promo unik atau nomor seri tertentu, *pure dense search* terkadang kalah presisi dibanding pencarian kata kunci eksak.
4. **Stateless (Tanpa Memori Percakapan):** Sistem memproses setiap *query* secara independen (*single-turn*) dan belum menyimpan riwayat percakapan (*chat history*) untuk pertanyaan lanjutan (*follow-up questions*).

---

## 5. Cara Menjalankan Notebook

### Prasyarat
* Akun Google dengan akses ke **Google Colab**.
* **Gemini API Key** aktif yang didapatkan dari [Google AI Studio](https://aistudio.google.com/).
* File dataset `ecommerce_faq_knowledge_base.csv`.

### Langkah-Langkah Eksekusi
1. **Buka Notebook:** Unggah dan buka file `Copy of starter_notebook_semantic_search.ipynb` di Google Colab.
2. **Unggah Dataset:** Buka panel **Files** (ikon folder di bilah kiri Colab), lalu unggah file `ecommerce_faq_knowledge_base.csv` ke direktori utama (`/content/`).
3. **Konfigurasi API Key (Secrets):**
   * Klik ikon **Secrets (Kunci)** di bilah navigasi kiri Colab.
   * Tambahkan *secret* baru dengan nama `GEMINI_API_KEY` dan masukkan nilai API key Anda.
   * Aktifkan *toggle* **Notebook access**.
4. **Jalankan Sel Berurutan:**
   * Klik menu **Runtime > Run all** (atau jalankan setiap *code cell* secara berurutan dari Section 1 hingga Section 8 menggunakan `Shift + Enter`).
   * Sel pertama akan otomatis menginstal pustaka yang dibutuhkan (`faiss-cpu` dan `google-genai`) sebelum membangun pipeline pencarian.


Overwriting README.md


In [12]:
%%writefile requirements.txt
pip install -r requirements.txt
pandas>=2.0.0
numpy>=1.24.0
faiss-cpu>=1.8.0
google-genai>=1.0.0

Overwriting requirements.txt
